# Australia Wildfire Activity Analysis
## Fifteen years of NASA satellite data — examined for risk patterns across seven regions

**Data:** NASA FIRMS / MODIS Collection 6 · **Period:** 2005–2020 · **Regions:** NSW, QL, SA, TA, VI, WA, NT

---

This notebook investigates three questions:

1. **How has wildfire severity changed over time — and is 2019–20 truly an outlier?**
2. **Which regions carry the most fire risk, and by how much?**
3. **How reliable is the satellite detection data at the extremes that matter most?**

Each section builds on the last. The full analytical report with narrative and context is available at [`Australia_Wildfire_Analysis_Report.html`](Australia_Wildfire_Analysis_Report.html).

---

## 1. Environment Setup

In [ ]:
# Uncomment to install if running outside the project environment
# %pip install -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import folium

%matplotlib inline

# Consistent visual style across all charts
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Libraries loaded.')

---
## 2. Data Loading & Preparation

The dataset is fetched directly from the NASA/IBM public URL — no local file required. Each row represents a daily aggregate of fire activity for one of Australia's seven regions, filtered to satellite observations with detection confidence above 75%.

In [ ]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv"

df = pd.read_csv(URL)

# Parse dates and extract time components for aggregation
df['Date']       = pd.to_datetime(df['Date'])
df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['Month_name'] = df['Date'].dt.strftime('%b')

print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
print(f'Period: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Regions: {sorted(df["Region"].unique())}')

In [ ]:
df.head()

In [ ]:
# Data types and null check
df.info()

In [ ]:
# Summary statistics — note the range of fire area and brightness values
df.describe().round(2)

---
## 3. How Has Wildfire Severity Changed Over Time?

Before comparing regions, we need to understand the overall trajectory. Is this a dataset with a stable baseline, a gradual trend, or discrete outlier events? The answer shapes how we interpret everything that follows.

### Annual fire area — identifying the outlier years

In [ ]:
df_year = df.groupby('Year')['Estimated_fire_area'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    df_year['Year'], df_year['Estimated_fire_area'],
    marker='o', linewidth=2.2, markersize=7,
    color='#C8390A', label='Annual average'
)
ax.fill_between(df_year['Year'], df_year['Estimated_fire_area'], alpha=0.12, color='#C8390A')

# Annotate the peak
peak = df_year.loc[df_year['Estimated_fire_area'].idxmax()]
ax.annotate(
    f"Peak: {peak['Estimated_fire_area']:.1f} km²",
    xy=(peak['Year'], peak['Estimated_fire_area']),
    xytext=(peak['Year'] - 2.5, peak['Estimated_fire_area'] * 0.92),
    arrowprops=dict(arrowstyle='->', color='#333'),
    fontsize=10, color='#333'
)

ax.set_title('Average Estimated Fire Area per Year — All Regions', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Avg Fire Area (km²)', fontsize=12)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('outputs/fire_area_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

> **Finding:** The 2019–20 season stands far above every prior year — this is not the continuation of a gradual trend, but a structural break. The Black Summer fires burned roughly 18.6 million hectares, a scale that the rest of the dataset's baseline cannot predict. Risk models that assume linear escalation will systematically underestimate tail exposure.

### Monthly seasonality — when fires consistently peak

In [ ]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df_month = df.groupby('Month_name')['Estimated_fire_area'].mean().reindex(month_order).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.bar(
    df_month['Month_name'], df_month['Estimated_fire_area'],
    color=plt.cm.Reds(np.linspace(0.3, 0.85, 12)),
    edgecolor='white', linewidth=0.8
)

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.5,
        f'{bar.get_height():.0f}',
        ha='center', va='bottom', fontsize=8, color='#444'
    )

ax.set_title('Average Estimated Fire Area by Month — Historical Average', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Avg Fire Area (km²)', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/fire_area_by_month.png', dpi=150, bbox_inches='tight')
plt.show()

> **Finding:** December, January, and February drive the majority of annual fire activity — every year, without exception. This U-shaped seasonal curve (high at year's start and end, low in the middle) is the southern hemisphere summer signature. The predictability of this window is itself actionable: October–November is a reliable preparation period before conditions deteriorate.

---
## 4. Which Regions Carry the Greatest Fire Risk?

Aggregate national statistics mask enormous regional variation. Here we break risk down into two dimensions: **intensity** (how hot fires burn, measured in Kelvin) and **spatial footprint** (how much area fires cover, measured in pixel count). A region can score high on one dimension without the other.

### Fire intensity by region — mean brightness with variability

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

sns.barplot(
    data=df,
    x='Region',
    y='Mean_estimated_fire_brightness',
    palette='YlOrRd',
    estimator='mean',
    errorbar='sd',
    ax=ax
)

# Reference line at the overall dataset mean
overall_mean = df['Mean_estimated_fire_brightness'].mean()
ax.axhline(
    overall_mean, color='steelblue', linestyle='--',
    linewidth=1.5, label=f'Dataset mean: {overall_mean:.1f} K'
)

ax.set_title('Mean Fire Brightness by Region (with standard deviation)', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Region', fontsize=12)
ax.set_ylabel('Mean Fire Brightness (Kelvin)', fontsize=12)
ax.tick_params(axis='x', rotation=10)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('outputs/brightness_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

> **Finding:** Regions above the dashed mean line burn consistently hotter than the national average. Error bars reveal that some regions are not only more intense on average — they are also more variable, meaning they produce both moderate and extreme events. That combination is the hardest risk profile to plan for.

### Spatial footprint — share of total fire detections by region

In [ ]:
region_counts = df.groupby('Region')['Count'].sum().sort_values(ascending=False)

colors  = plt.cm.Oranges(np.linspace(0.4, 0.92, len(region_counts)))
explode = [0.06 if i == 0 else 0 for i in range(len(region_counts))]

fig, ax = plt.subplots(figsize=(9, 9))

wedges, _, autotexts = ax.pie(
    region_counts,
    labels=None,
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    explode=explode,
    shadow=True,
    pctdistance=0.82,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)

for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax.legend(
    wedges,
    [f'{r}  —  {c:,} pixels' for r, c in zip(region_counts.index, region_counts.values)],
    title='Region · Total Pixels Detected',
    loc='lower left',
    bbox_to_anchor=(-0.05, -0.05),
    fontsize=10,
    title_fontsize=11
)

ax.set_title('Share of Total Fire Pixel Detections by Region', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('outputs/pixel_count_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

> **Finding:** Two regions account for the majority of all fire pixel detections across the full dataset. Risk is not distributed evenly — it is heavily concentrated geographically. Tasmania and Victoria show the smallest footprints, though their fire events carry outsized human impact due to population density and proximity to urban areas.

### Brightness distribution by region — understanding the full range, not just the average

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(
    data=df,
    x='Mean_estimated_fire_brightness',
    hue='Region',
    multiple='stack',
    palette='tab10',
    bins=30,
    edgecolor='white',
    linewidth=0.5,
    ax=ax
)

ax.set_title('Fire Brightness Distribution by Region — Stacked', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Mean Estimated Fire Brightness (Kelvin)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/brightness_distribution_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

> **Finding:** The stacked view separates two things the bar chart averaged together: where the bulk of observations sit, and which regions contribute to the high-temperature tail. Regions that appear moderate on average may still contribute meaningfully to extreme events — visible here as their color appearing in the right tail of the distribution.

### Geographic overview — all seven regions on the map

In [ ]:
region_data = {
    'region':   ['NSW', 'QL',  'SA',   'TA',   'VI',   'WA',   'NT'],
    'Lat':      [-31.88, -22.16, -30.53, -42.04, -36.60, -25.23, -19.49],
    'Lon':      [147.29, 144.58, 135.63, 146.64, 144.68, 121.02, 132.55],
    'FullName': [
        'New South Wales', 'Queensland', 'South Australia',
        'Tasmania', 'Victoria', 'Western Australia', 'Northern Territory'
    ]
}
reg = pd.DataFrame(region_data)

# Pull aggregated stats per region to display in popups
region_stats = df.groupby('Region').agg(
    Avg_Fire_Area  = ('Estimated_fire_area',            'mean'),
    Avg_Brightness = ('Mean_estimated_fire_brightness', 'mean'),
    Total_Pixels   = ('Count',                          'sum')
).reset_index()

aus_map = folium.Map(location=[-25, 135], zoom_start=4, tiles='CartoDB positron')

for _, row in reg.iterrows():
    r_code    = row['region']
    stats_row = region_stats[region_stats['Region'] == r_code]

    if not stats_row.empty:
        s = stats_row.iloc[0]
        popup_html = f"""
            <b style='font-size:13px'>{row['FullName']} ({r_code})</b><br><br>
            Avg Fire Area: <b>{s['Avg_Fire_Area']:.1f} km²</b><br>
            Avg Brightness: <b>{s['Avg_Brightness']:.1f} K</b><br>
            Total Pixels: <b>{int(s['Total_Pixels']):,}</b>
        """
    else:
        popup_html = f"<b>{row['FullName']} ({r_code})</b>"

    folium.CircleMarker(
        location     = [row['Lat'], row['Lon']],
        radius       = 18,
        popup        = folium.Popup(popup_html, max_width=220),
        tooltip      = f"{row['FullName']} ({r_code})",
        color        = '#8B0000',
        fill         = True,
        fill_color   = '#FF4500',
        fill_opacity = 0.65,
        weight       = 2
    ).add_to(aus_map)

    folium.Marker(
        location = [row['Lat'], row['Lon']],
        icon     = folium.DivIcon(
            html       = f'<div style="font-size:11px;font-weight:bold;color:white;text-align:center;margin-top:5px">{r_code}</div>',
            icon_size  = (40, 20),
            icon_anchor= (20, 10)
        )
    ).add_to(aus_map)

aus_map.save('outputs/australia_wildfire_map.html')
aus_map

Click any marker to see the region's average fire area, brightness, and total pixel count across the full dataset period.

---
## 5. How Reliable Is the Detection Data?

Before treating satellite data as ground truth, it's worth asking whether detection quality varies systematically with fire intensity. If the sensor detects intense fires more reliably than mild ones, that affects how we interpret low-end observations.

### Overall brightness distribution — checking for skew

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(
    df['Mean_estimated_fire_brightness'],
    bins=30,
    color='#C8390A',
    edgecolor='white',
    alpha=0.85,
    linewidth=0.8
)

mean_val   = df['Mean_estimated_fire_brightness'].mean()
median_val = df['Mean_estimated_fire_brightness'].median()

ax.axvline(mean_val,   color='steelblue', linestyle='--', linewidth=2,
           label=f'Mean: {mean_val:.1f} K')
ax.axvline(median_val, color='seagreen',  linestyle=':',  linewidth=2,
           label=f'Median: {median_val:.1f} K')

ax.set_title('Distribution of Mean Estimated Fire Brightness', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Mean Fire Brightness (Kelvin)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('outputs/brightness_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

> **Finding:** The mean sits above the median, confirming a right skew — most observations cluster around a moderate temperature, with a long right tail of rare, high-intensity events. This distribution shape matters for modelling: a normal distribution assumption would underestimate the frequency of extreme events.

### Radiative power vs. detection confidence — does intensity predict reliability?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x='Mean_confidence',
    y='Mean_estimated_fire_radiative_power',
    hue='Region',
    palette='tab10',
    alpha=0.50,
    s=30,
    ax=ax
)

# OLS trend line
m, b    = np.polyfit(df['Mean_confidence'].dropna(),
                     df['Mean_estimated_fire_radiative_power'].dropna(), 1)
x_range = np.linspace(df['Mean_confidence'].min(), df['Mean_confidence'].max(), 100)
ax.plot(x_range, m * x_range + b, color='#1a1a1a', linewidth=2,
        linestyle='--', label='OLS trend')

# Pearson r
corr = df[['Mean_confidence', 'Mean_estimated_fire_radiative_power']].corr().iloc[0, 1]
ax.text(0.05, 0.93, f'Pearson r = {corr:.3f}', transform=ax.transAxes,
        fontsize=11, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85))

ax.set_title('Radiative Power vs. Detection Confidence by Region', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Mean Confidence (%)', fontsize=12)
ax.set_ylabel('Mean Radiative Power (MW)', fontsize=12)
ax.legend(title='Region', fontsize=9, title_fontsize=10, loc='upper right')
plt.tight_layout()
plt.savefig('outputs/scatter_power_vs_confidence.png', dpi=150, bbox_inches='tight')
plt.show()

> **Finding:** Pearson r ≈ +0.28 — a weak positive correlation. More intense fires are detected with slightly higher confidence, which is a reassuring property: the data is marginally more reliable at the extremes where precision matters most. However, confidence is not a strong enough signal to use as a quality filter without losing too much data.

---
## 6. Interactive Dashboard

The static charts above answer fixed questions. The Dash dashboard below allows free exploration — compare any region's fire season against any year, or watch how monthly patterns shift between 2005 and 2020.

Running the cell saves `app.py` to the project root. Launch it from the terminal with `python app.py`, then open `http://127.0.0.1:8050`.

In [ ]:
app_code = '''
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# ── Data ─────────────────────────────────────────────────────────
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv"
df = pd.read_csv(URL)
df['Date']  = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

regions = sorted(df['Region'].unique())
years   = sorted(df['Year'].unique())

MONTH_LABELS = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

# ── Layout ───────────────────────────────────────────────────────
app = Dash(__name__)

app.layout = html.Div([

    html.H1(
        'Australia Wildfire Activity Dashboard',
        style={'textAlign':'center','color':'#B22222',
               'fontFamily':'Georgia, serif','marginBottom':'4px','paddingTop':'24px'}
    ),
    html.P(
        'Select a region and year to explore monthly fire patterns.',
        style={'textAlign':'center','color':'#666','fontFamily':'sans-serif','marginBottom':'24px'}
    ),

    html.Div([
        html.Div([
            html.Label('Region', style={'fontWeight':'600','fontFamily':'sans-serif','fontSize':'13px'}),
            dcc.RadioItems(
                id='region-radio',
                options=[{'label': r, 'value': r} for r in regions],
                value=regions[0],
                inline=True,
                labelStyle={'marginRight':'16px','fontFamily':'sans-serif','fontSize':'14px'}
            ),
        ], style={'marginBottom':'14px'}),
        html.Div([
            html.Label('Year', style={'fontWeight':'600','fontFamily':'sans-serif','fontSize':'13px'}),
            dcc.Dropdown(
                id='year-dropdown',
                options=[{'label': str(y), 'value': y} for y in years],
                value=years[-1],
                clearable=False,
                style={'width':'160px','fontFamily':'sans-serif'}
            ),
        ]),
    ], style={
        'background':'#FFF8F0','padding':'20px 30px',
        'borderRadius':'6px','margin':'0 30px 20px',
        'boxShadow':'0 2px 6px rgba(0,0,0,0.08)'
    }),

    html.Div([
        html.Div(id='pie-container', style={'width':'48%','display':'inline-block'}),
        html.Div(id='bar-container', style={'width':'48%','display':'inline-block','float':'right'}),
    ], style={'margin':'0 30px','overflow':'hidden'}),

], style={'background':'#FFFAF7','minHeight':'100vh','paddingBottom':'40px'})


# ── Callback ─────────────────────────────────────────────────────
@app.callback(
    Output('pie-container', 'children'),
    Output('bar-container', 'children'),
    Input('region-radio',   'value'),
    Input('year-dropdown',  'value')
)
def update(region, year):
    filtered = df[(df['Region'] == region) & (df['Year'] == year)]

    monthly = filtered.groupby('Month').agg(
        Avg_Fire_Area = ('Estimated_fire_area', 'mean'),
        Avg_Count     = ('Count', 'mean')
    ).reset_index()
    monthly['Month_Name'] = monthly['Month'].map(MONTH_LABELS)

    pie = px.pie(
        monthly,
        names='Month_Name', values='Avg_Fire_Area',
        title=f'Monthly Avg Fire Area — {region} ({year})',
        color_discrete_sequence=px.colors.sequential.Reds_r,
        hole=0.35
    )
    pie.update_traces(textposition='inside', textinfo='percent+label')
    pie.update_layout(title_font_size=13, margin=dict(t=48,b=8,l=8,r=8),
                      paper_bgcolor='#FFFAF7')

    bar = px.bar(
        monthly.sort_values('Month'),
        x='Month_Name', y='Avg_Count',
        title=f'Monthly Avg Pixel Count — {region} ({year})',
        color='Avg_Count', color_continuous_scale='OrRd',
        labels={'Avg_Count':'Avg Pixels','Month_Name':'Month'}
    )
    bar.update_layout(title_font_size=13, margin=dict(t=48,b=8,l=8,r=8),
                      paper_bgcolor='#FFFAF7', coloraxis_showscale=False)

    return dcc.Graph(figure=pie), dcc.Graph(figure=bar)


if __name__ == '__main__':
    app.run(debug=True)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print('app.py saved. Run with: python app.py')

---
## 7. Summary of Findings

| Question | Finding |
|----------|---------|
| How has severity changed over time? | 2019–20 is a structural break, not a trend — fire area exceeded any prior year by a wide margin. Seasonality (Dec–Feb peak) is stable and reliable every year. |
| Which regions carry the most risk? | Queensland and the Northern Territory dominate on both intensity and spatial footprint, accounting for the majority of all fire pixel detections. Tasmania and Victoria have the smallest footprint but highest population exposure. |
| How reliable is the detection data? | A weak positive correlation (r ≈ +0.28) between radiative power and confidence means data quality is marginally better at the extremes. Fire brightness is right-skewed — normal distribution assumptions will underestimate tail events. |

---

For the full narrative report with context and implications, see [`Australia_Wildfire_Analysis_Report.html`](Australia_Wildfire_Analysis_Report.html).